In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [2]:
ocr_dir = Path("results/mol_rep_ocr/v1.1/ablation_num_train_epoch")
conversion_dir = Path("results/mol_rep_conversion/v1.1/ablation_num_train_epoch")

## OCR

In [3]:
ocr_raw_reponses = pd.DataFrame()

for file in (ocr_dir / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    ocr_raw_reponses = pd.concat([ocr_raw_reponses, df], axis=0, ignore_index=True)

In [4]:
from src.utils import compute_mol_metrics

ocr_df = ocr_raw_reponses.copy()

ocr_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = ocr_raw_reponses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [5]:
def aggregate_func(df):
    return pd.Series({
        "num_samples": df.shape[0],
        "gt_valid_ratio": df["is_gt_valid"].mean(),
        "pred_valid_ratio": df["is_pred_valid"].mean(),
        "em_ratio": df["is_em"].mean(),
        "can_smiles_match_ratio": df["is_can_smiles_match"].mean(),
        "inchikey_match_ratio": df["is_inchikey_match"].mean(),
        "tanimoto_sim_mean": df["tanimoto_sim"].mean(),
        "tanimoto_sim_std": df["tanimoto_sim"].std(),
    })

In [6]:
ocr_df.groupby(["model_name", "num_train_epoch"]).apply(aggregate_func).to_csv(ocr_dir / "metrics_overall.csv")
ocr_df.groupby(["model_name", "num_train_epoch"]).apply(aggregate_func)

num_samples  \
model_name                                         num_train_epoch                
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... 1                       80.0   
                                                   2                       80.0   
                                                   3                       80.0   
                                                   4                       80.0   
                                                   5                       80.0   
                                                   6                       80.0   
                                                   7                       80.0   
qwen3_vl_4b_i_sft_lora_conversion                  1                       80.0   
                                                   2                       80.0   
                                                   3                       80.0   
                                                   4                       80.0   
                                                   5                       80.0   
                                                   6                       80.0   
                                                   7                       80.0   
                                                   8                       80.0   
                                                   9                       80.0   
                                                   10                      80.0   
qwen3_vl_4b_i_sft_lora_ocr                         1                       80.0   
                                                   2                       80.0   
                                                   3                       80.0   
                                                   4                       80.0   
                                                   5                       80.0   
                                                   6                       80.0   
qwen3_vl_4b_i_sft_lora_ocr_conversion              1                       80.0   
                                                   2                       80.0   
                                                   3                       80.0   
                                                   4                       80.0   
                                                   5                       80.0   
                                                   6                       80.0   
                                                   7                       80.0   
                                                   8                       80.0   
                                                   9                       80.0   
                                                   10                      80.0   

                                                                    gt_valid_ratio  \
model_name                                         num_train_epoch                   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... 1                           1.0   
                                                   2                           1.0   
                                                   3                           1.0   
                                                   4                           1.0   
                                                   5                           1.0   
                                                   6                           1.0   
                                                   7                           1.0   
qwen3_vl_4b_i_sft_lora_conversion                  1                           1.0   
                                                   2                           1.0   
                                                   3                           1.0   
                                                   4                           1.0   
                                            

In [7]:
# ocr_df.groupby("_parse_status").apply(aggregate_func).to_csv(ocr_dir / "metrics_by_parse_status.csv")
ocr_df[ocr_df._parse_status == "success"].groupby(["model_name", "num_train_epoch"]).apply(aggregate_func)

num_samples  \
model_name                                         num_train_epoch                
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... 1                       54.0   
                                                   2                       56.0   
                                                   3                       55.0   
                                                   4                       54.0   
                                                   5                       56.0   
                                                   6                       56.0   
                                                   7                       54.0   
qwen3_vl_4b_i_sft_lora_conversion                  1                       42.0   
                                                   2                       46.0   
                                                   3                       46.0   
                                                   4                       49.0   
                                                   5                       51.0   
                                                   6                       48.0   
                                                   7                       55.0   
                                                   8                       51.0   
                                                   9                       54.0   
                                                   10                      54.0   
qwen3_vl_4b_i_sft_lora_ocr                         1                       18.0   
                                                   2                       18.0   
                                                   3                       26.0   
                                                   4                       35.0   
                                                   5                       31.0   
                                                   6                       33.0   
qwen3_vl_4b_i_sft_lora_ocr_conversion              1                       40.0   
                                                   2                       43.0   
                                                   3                       45.0   
                                                   4                       48.0   
                                                   5                       48.0   
                                                   6                       51.0   
                                                   7                       50.0   
                                                   8                       54.0   
                                                   9                       54.0   
                                                   10                      54.0   

                                                                    gt_valid_ratio  \
model_name                                         num_train_epoch                   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... 1                           1.0   
                                                   2                           1.0   
                                                   3                           1.0   
                                                   4                           1.0   
                                                   5                           1.0   
                                                   6                           1.0   
                                                   7                           1.0   
qwen3_vl_4b_i_sft_lora_conversion                  1                           1.0   
                                                   2                           1.0   
                                                   3                           1.0   
                                                   4                           1.0   
                                            

## Conversion

In [8]:
conversion_raw_reponses = pd.DataFrame()

for file in (conversion_dir / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    conversion_raw_reponses = pd.concat([conversion_raw_reponses, df], axis=0, ignore_index=True)

In [9]:
conversion_df = conversion_raw_reponses.copy()

conversion_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = conversion_raw_reponses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [10]:
conversion_df.groupby(["model_name", "num_train_epoch"]).apply(aggregate_func).to_csv(conversion_dir / "metrics_overall.csv")
conversion_df.groupby(["model_name", "num_train_epoch"]).apply(aggregate_func)

num_samples  \
model_name                                         num_train_epoch                
qwen3_4b_i_sft_lora_conversion                     1                      875.0   
                                                   2                      875.0   
                                                   3                      875.0   
                                                   4                      875.0   
                                                   5                      875.0   
                                                   6                      875.0   
                                                   7                      875.0   
                                                   8                      875.0   
                                                   9                      875.0   
                                                   10                     875.0   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... 1                      875.0   
                                                   2                      875.0   
                                                   3                      875.0   
                                                   4                      875.0   
                                                   5                      875.0   
                                                   6                      875.0   
                                                   7                      875.0   
qwen3_vl_4b_i_sft_lora_conversion                  1                      875.0   
                                                   2                      875.0   
                                                   3                      875.0   
                                                   4                      875.0   
                                                   5                      875.0   
                                                   6                      875.0   
                                                   7                      875.0   
                                                   8                      875.0   
                                                   9                      875.0   
                                                   10                     875.0   
qwen3_vl_4b_i_sft_lora_ocr                         1                      875.0   
                                                   2                      875.0   
                                                   3                      875.0   
                                                   4                      875.0   
                                                   5                      875.0   
                                                   6                      875.0   
qwen3_vl_4b_i_sft_lora_ocr_conversion              1                      875.0   
                                                   2                      875.0   
                                                   3                      875.0   
                                                   4                      875.0   
                                                   5                      875.0   
                                                   6                      875.0   
                                                   7                      875.0   
                                                   8                      875.0   
                                                   9                      875.0   
                                                   10                     875.0   

                                                                    gt_valid_ratio  \
model_name                                         num_train_epoch                   
qwen3_4b_i_sft_lora_conversion                     1                           1.0   
                                                   2                      

In [11]:
# conversion_df.groupby("_parse_status").apply(aggregate_func).to_csv(conversion_dir / "metrics_by_parse_status.csv")
conversion_df[conversion_df._parse_status == "success"].groupby(["model_name", "num_train_epoch"]).apply(aggregate_func)

num_samples  \
model_name                                         num_train_epoch                
qwen3_4b_i_sft_lora_conversion                     1                      342.0   
                                                   2                      423.0   
                                                   3                      522.0   
                                                   4                      570.0   
                                                   5                      665.0   
                                                   6                      729.0   
                                                   7                      757.0   
                                                   8                      767.0   
                                                   9                      791.0   
                                                   10                     789.0   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... 1                      797.0   
                                                   2                      798.0   
                                                   3                      795.0   
                                                   4                      790.0   
                                                   5                      791.0   
                                                   6                      793.0   
                                                   7                      794.0   
qwen3_vl_4b_i_sft_lora_conversion                  1                      406.0   
                                                   2                      469.0   
                                                   3                      548.0   
                                                   4                      612.0   
                                                   5                      671.0   
                                                   6                      701.0   
                                                   7                      759.0   
                                                   8                      786.0   
                                                   9                      787.0   
                                                   10                     786.0   
qwen3_vl_4b_i_sft_lora_ocr                         1                      161.0   
                                                   2                      158.0   
                                                   3                      191.0   
                                                   4                      217.0   
                                                   5                      209.0   
                                                   6                      196.0   
qwen3_vl_4b_i_sft_lora_ocr_conversion              1                      428.0   
                                                   2                      486.0   
                                                   3                      556.0   
                                                   4                      645.0   
                                                   5                      681.0   
                                                   6                      734.0   
                                                   7                      781.0   
                                                   8                      773.0   
                                                   9                      792.0   
                                                   10                     802.0   

                                                                    gt_valid_ratio  \
model_name                                         num_train_epoch                   
qwen3_4b_i_sft_lora_conversion                     1                           1.0   
                                                   2                      

In [12]:
# conversion_df.groupby(["output_rep_type"]).apply(aggregate_func).to_csv(conversion_dir / "metrics_by_output_rep_type.csv")
# conversion_df.groupby(["model_name", "output_rep_type"]).apply(aggregate_func)

In [13]:
# conversion_df.groupby(["input_rep_type"]).apply(aggregate_func)

In [14]:
# conversion_df.groupby(["task_type"]).apply(aggregate_func).to_csv(conversion_dir / "metrics_by_task_type.csv")
# conversion_df.groupby(["model_name", "task_type"]).apply(aggregate_func)

In [15]:
# conversion_df.groupby(["model_name", "task_id"]).apply(aggregate_func)